# 02 Preprocessing Dataset Final

Notebook ini membangun dataset final untuk klasifikasi nilai pasar pemain Big 5 Eropa. Sumber data yang dipakai adalah hasil scraping Transfermarkt yang sudah tersedia dan hasil scraping FBref yang sudah tersedia. Notebook ini tidak menjalankan scraping Transfermarkt ulang.

## Tujuan Preprocessing

Tahap ini melakukan pembersihan data Transfermarkt, membuat fitur historis yang aman, menggabungkan fitur performa pemain dari FBref, dan menyimpan dataset siap training ke path utama.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

RAW_TRANSFERMARKT_FILE = PROJECT_ROOT / "data" / "raw" / "players_raw.csv"
FBREF_FILE = PROJECT_ROOT / "data" / "interim" / "fbref_player_stats.csv"
MODEL_OUTPUT_FILE = PROJECT_ROOT / "data" / "model" / "players_model.csv"
CLEAN_OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / "transfermarkt_dataset_clean.csv"

for path in [RAW_TRANSFERMARKT_FILE, FBREF_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"File input tidak ditemukan: {path}")

RAW_TRANSFERMARKT_FILE, FBREF_FILE, CLEAN_OUTPUT_FILE, MODEL_OUTPUT_FILE

## Build Dataset Final

Fungsi `build_final_dataset` membaca raw Transfermarkt yang sudah ada, membuat dataset clean, lalu menggabungkan FBref memakai entity matching bertahap. Matching dilakukan dari yang paling ketat sampai fallback yang hanya dipakai jika unik di kedua dataset.

In [ ]:
from src.preprocessing.clean_dataset import build_final_dataset, save_outputs

clean_df, model_df, preprocessing_report = build_final_dataset()
processed_file, model_file = save_outputs(clean_df, model_df)
preprocessing_report

## Ringkasan Dataset

In [ ]:
print(f"Clean dataset shape: {clean_df.shape}")
print(f"Model dataset shape: {model_df.shape}")
print(f"Clean output: {processed_file}")
print(f"Model output: {model_file}")

In [ ]:
model_df.head()

## Distribusi Label

In [ ]:
model_df["market_value_category"].value_counts()

In [ ]:
model_df["market_value_category"].value_counts(normalize=True).round(3)

## Audit Matching FBref

In [ ]:
import pandas as pd

matching = pd.read_csv("data/interim/player_matching_result.csv")
unmatched = pd.read_csv("data/interim/unmatched_players.csv")

matching["match_method"].value_counts()

In [ ]:
model_df["has_performance_stats"].value_counts()

In [ ]:
model_df["has_performance_stats"].value_counts(normalize=True).round(3)

In [ ]:
unmatched[["player_name", "club", "league", "season"]].head(20)

## Cek Fitur Performa

In [ ]:
performance_columns = preprocessing_report["performance_columns"]
performance_columns

In [ ]:
model_df[performance_columns].describe().T

## Cek Fitur Terlarang

Fitur yang tidak tersedia dari hasil FBref aktual tidak boleh masuk dataset training.

In [ ]:
forbidden_features = {
    "xg",
    "non_penalty_xg",
    "xg_per_90",
    "non_penalty_xg_per_90",
    "aerial_won",
    "aerial_lost",
    "ball_recoveries",
}

sorted(forbidden_features & set(model_df.columns))

## Output Preprocessing

Jika semua cell di atas berjalan tanpa error, lanjutkan ke `03_training_model.ipynb`.